# Experimento especial v1.5 — Pesquisas repetidas e cobertura de 95%

Atividade da Unidade 05. Usa candidatos **inteiramente fictícios** e uma população sintética de 1 milhão de eleitores. Não reproduz pesquisas efetivas de Quaest, Datafolha ou AtlasIntel. O objetivo é compreender **quando a frequência de intervalos corretos se aproxima de 95%** e quando isso não ocorre.

Execute as células em ordem. O primeiro experimento roda sem arquivos externos; para o painel completo dos cinco procedimentos, envie o arquivo `rodadas_cobertura_por_candidato_v1_5.csv.gz` incluído no pacote.

### Experimento especial: repetição de pesquisas e cobertura de 95%

A população sintética de **1 milhão de eleitores** (v0.7/v0.8/v1.1) possui preferências fictícias conhecidas: **A = 41,9945%**, **B = 38,0190%**, **C = 12,0219%** e outras respostas. Podemos repetir milhares de pesquisas sobre **a mesma população fixa**.

Cada pesquisa produz, para cada candidato, uma **estimativa pontual** e um **intervalo de confiança**. Contamos quantos intervalos contêm a proporção verdadeira da população. Após \(R\) pesquisas, a **cobertura acumulada** é

\[
\widehat{C}_R=\frac{1}{R}\sum_{r=1}^{R}\mathbf{1}\{L_r\leq p_{\mathrm{verdadeiro}}\leq U_r\}.
\]

Se o intervalo for calibrado para o desenho e as hipóteses forem satisfeitas, essa frequência tende à **cobertura real do procedimento**, idealmente próxima de 95%. **Não se espera uma aproximação monotônica nem uma cobertura exatamente igual a 95%.** Em um método enviesado, a frequência pode convergir para um valor muito inferior.

Os nomes *domiciliar*, *pontos de fluxo* e *digital* representam **modelos didáticos**, não reconstituições dos procedimentos de Quaest, Datafolha ou AtlasIntel. Nos métodos por cotas e digitais, os intervalos são *pseudo-intervalos* cuja cobertura apenas observamos, sem garantia nominal.

In [ ]:
# Execução autossuficiente: controle AAS e hipótese simplificada de não resposta.
# Para comparar também os cinco métodos ancorados na população de 1 milhão,
# execute a célula seguinte com o CSV de rodadas do pacote v1.5.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CANDIDATOS = ["A", "B", "C", "OUTRA", "INDECISO", "BRANCO_NULO"]
VERDADE = np.array([0.419945, 0.380190, 0.120219, 0.029968, 0.029766, 0.019912])
assert abs(VERDADE.sum()-1) < 1e-12
NUM_REPETICOES = 3000
ENTREVISTAS_POR_PESQUISA = 2000
rng_cobertura = np.random.default_rng(202610161015)

def intervalo_wilson(contagens, n, z=1.959963984540054):
    prop = contagens / n
    den = 1 + z*z/n
    centro = (prop + z*z/(2*n)) / den
    semi = z / den * np.sqrt(prop*(1-prop)/n + z*z/(4*n*n))
    return centro-semi, centro+semi

cenarios_didaticos = {}
for rotulo, fator_A in [("AAS idealizada", 1.0), ("Resposta seletiva: A × 0,55", 0.55)]:
    prob = VERDADE * np.array([fator_A, 1, 1, 1, 1, 1])
    prob /= prob.sum()
    contagens = rng_cobertura.multinomial(ENTREVISTAS_POR_PESQUISA, prob, size=NUM_REPETICOES)
    baixo, alto = intervalo_wilson(contagens, ENTREVISTAS_POR_PESQUISA)
    cobertura = (baixo <= VERDADE) & (VERDADE <= alto)
    cenarios_didaticos[rotulo] = {"estimativas":contagens/ENTREVISTAS_POR_PESQUISA,
                                  "cobertura":cobertura}

fig, ax = plt.subplots(figsize=(10, 5))
R = np.arange(1, NUM_REPETICOES+1)
for nome, dados in cenarios_didaticos.items():
    frequencia = 100 * np.cumsum(dados["cobertura"][:, 0]) / R
    ax.plot(R[19:], frequencia[19:], label=nome)
ax.axhline(95, linestyle="--", color="black", label="referência: 95%")
ax.set(xlabel="Número de pesquisas repetidas", ylabel="Intervalos que contêm a verdade (%)",
       title="Cobertura acumulada para o candidato fictício A")
ax.set_ylim(-2, 102)
ax.legend()
ax.grid(alpha=.2)
plt.show()
print("Coberturas finais do candidato A:")
for nome, dados in cenarios_didaticos.items():
    print(f"{nome}: {100*dados['cobertura'][:,0].mean():.2f}%")

#### Cinco procedimentos sintéticos, candidato por candidato

O CSV v1.5 contém **3.000 pesquisas por método e cenário**, todas com 2.000 entrevistas e calculadas sobre a população fixa de 1 milhão de eleitores. Se o CSV ainda não está no diretório `dados/` do repositório, coloque-o no Colab pelo painel de arquivos ou faça upload manual. A célula anterior funciona independentemente desse arquivo.

Para examinar a convergência, altere `CANDIDATO_EXAMINADO`, `FATOR_RESPOSTA_A` e `REPETICOES_VISIVEIS` e execute novamente a célula abaixo. A linha horizontal de 95% é **alvo teórico para os controles probabilísticos**, não uma meta garantida para todas as curvas.

In [ ]:
from pathlib import Path
from IPython.display import display

ARQUIVO_COBERTURA = next((p for p in [
    Path('dados/rodadas_cobertura_por_candidato_v1_5.csv.gz'),
    Path('rodadas_cobertura_por_candidato_v1_5.csv.gz'),
    Path('/content/rodadas_cobertura_por_candidato_v1_5.csv.gz'),
] if p.is_file()), None)

# Configuração do gráfico: altere e execute novamente.
CANDIDATO_EXAMINADO = 'A'         # A, B, C, OUTRA, INDECISO, BRANCO_NULO
FATOR_RESPOSTA_A = 1.00         # 1.00 (neutro) ou 0.55 (participação seletiva)
REPETICOES_VISIVEIS = 3000      # de 30 a 3000

rotulos = {
    'aleatoria_simples':'Aleatória simples (IC probabilístico)',
    'municipal_probabilistico':'Municípios PPS (IC por conglomerados)',
    'domiciliar_seletivo':'Domiciliar (pseudo-IC)',
    'fluxo_cotas_aproximado':'Fluxo com cotas, aproximação (pseudo-IC)',
    'digital_posestratificado':'Digital pós-estratificado (pseudo-IC)',
}
if ARQUIVO_COBERTURA is None:
    print('Arquivo de rodadas v1.5 não encontrado. Para os cinco métodos,')
    print('envie rodadas_cobertura_por_candidato_v1_5.csv.gz ao Colab.')
    print('O primeiro gráfico, de simulação autossuficiente, já está disponível acima.')
else:
    rodadas_95 = pd.read_csv(ARQUIVO_COBERTURA)
    assert 30 <= REPETICOES_VISIVEIS <= 3000
    d95 = rodadas_95.loc[
        (rodadas_95['candidato'] == CANDIDATO_EXAMINADO) &
        np.isclose(rodadas_95['cenario_resposta_A'], FATOR_RESPOSTA_A) &
        (rodadas_95['rep'] <= REPETICOES_VISIVEIS)
    ].copy()
    assert len(d95) == 5 * REPETICOES_VISIVEIS
    fig, ax = plt.subplots(figsize=(11, 5.5))
    for metodo, grupo in d95.groupby('metodo'):
        grupo = grupo.sort_values('rep')
        ax.plot(grupo['rep'].to_numpy()[19:],
                100*grupo['cobertura_acumulada'].to_numpy()[19:],
                linestyle='-' if metodo in ('aleatoria_simples','municipal_probabilistico') else '--',
                label=rotulos[metodo])
    ax.axhline(95, color='black', linestyle=':', linewidth=2,
               label='95% — referência nominal')
    ax.set(xlabel='Número de pesquisas repetidas',
           ylabel='Cobertura acumulada (%)',
           title=f'Candidato {CANDIDATO_EXAMINADO} — fator de resposta A: {FATOR_RESPOSTA_A:.2f}')
    ax.set_ylim(-3,103)
    ax.grid(alpha=.2)
    ax.legend(fontsize=8, ncol=2)
    plt.show()
    print('Cobertura até a repetição', REPETICOES_VISIVEIS)
    display(d95.groupby('metodo',as_index=False).agg(
        cobertura_empirica=('cobre', lambda s:100*s.mean()),
        estimativa_media=('estimativa',lambda s:100*s.mean()),
        valor_verdadeiro=('verdade',lambda s:100*s.iloc[0])
    ).round(2))

**Perguntas para os estudantes:** (1) Por que a curva do controle aleatório oscila muito nas primeiras pesquisas? (2) A partir de quantas repetições ela se estabiliza aproximadamente? (3) Por que uma curva pode se estabilizar **longe de 95%**, mesmo com milhares de pesquisas? (4) O que muda quando selecionamos o candidato B ou C? (5) Por que a pós-estratificação demográfica não corrige necessariamente a não resposta dependente do voto?

**Precisão:** em 3.000 repetições, se a cobertura verdadeira é 95%, o desvio-padrão da frequência simulada é de aproximadamente **0,40 ponto percentual**. Por isso 94,8%, 95,2% ou 95,5% não são anomalias. Não é a **estimativa de votos** que tende a 95%: é a proporção de **intervalos contendo a verdade** que tende à cobertura real do método.